<!-- ######
     This file was developed with the assistance of Claude by Anthropic.
     ###### -->

# Demo : conversion des codes WMO en icones MeteoSwiss

Ce notebook montre concretement comment MeteoGlobe transforme un **code meteo WMO** (renvoye par Open-Meteo) en un **numero d'icone MeteoSwiss** (un fichier SVG sur le disque du serveur).

Tout le code ici est une **traduction en Python** de ce que fait le frontend dans `public/app.js`. Le comportement est exactement le meme : tu peux executer chaque cellule et voir le resultat.

## Plan du notebook

1. La table de correspondance `WMO_TO_METEO`
2. La fonction `get_meteo_icon()` (= `getMeteoIcon()` en JS)
3. Cas simple : un code connu
4. Cas "plusieurs codes -> meme icone" (les trois bruines)
5. Cas "code inconnu -> fallback ciel degage"
6. La logique jour / nuit (`is_day`)
7. Exemple complet : on simule une reponse Open-Meteo et on affiche l'icone choisie

## 1. La table `WMO_TO_METEO`

C'est un dictionnaire qui associe a chaque **code WMO** (la cle) un dictionnaire `{ d, n }` :

- `d` = numero d'icone a afficher **le jour**
- `n` = numero d'icone a afficher **la nuit**

Pour certaines situations (ciel degage, peu nuageux...), `d` et `n` sont **differents** parce que l'icone montre soit un soleil, soit une lune. Pour les autres (pluie, brouillard, orage...), `d` et `n` sont **identiques** parce que ni le soleil ni la lune n'apparaissent dans le dessin.

In [ ]:
WMO_TO_METEO = {
    0:  {"d": 1,  "n": 101},  # Ciel degage
    1:  {"d": 2,  "n": 102},  # Peu nuageux
    2:  {"d": 3,  "n": 103},  # Partiellement nuageux
    3:  {"d": 5,  "n": 105},  # Couvert
    45: {"d": 25, "n": 25},   # Brouillard
    48: {"d": 25, "n": 25},   # Brouillard givrant
    51: {"d": 7,  "n": 7},    # Bruine legere
    53: {"d": 7,  "n": 7},    # Bruine moderee
    55: {"d": 7,  "n": 7},    # Bruine dense
    56: {"d": 26, "n": 26},   # Bruine verglacante legere
    57: {"d": 26, "n": 26},   # Bruine verglacante dense
    61: {"d": 7,  "n": 7},    # Pluie legere
    63: {"d": 8,  "n": 8},    # Pluie moderee
    65: {"d": 9,  "n": 9},    # Pluie forte
    66: {"d": 26, "n": 26},   # Pluie verglacante legere
    67: {"d": 26, "n": 26},   # Pluie verglacante forte
    71: {"d": 13, "n": 13},   # Neige legere
    73: {"d": 14, "n": 14},   # Neige moderee
    75: {"d": 15, "n": 15},   # Neige forte
    77: {"d": 15, "n": 15},   # Grains de neige
    80: {"d": 17, "n": 17},   # Averses legeres
    81: {"d": 18, "n": 18},   # Averses moderees
    82: {"d": 19, "n": 19},   # Averses violentes
    85: {"d": 17, "n": 17},   # Averses de neige legeres
    86: {"d": 15, "n": 15},   # Averses de neige fortes
    95: {"d": 23, "n": 23},   # Orage
    96: {"d": 20, "n": 20},   # Orage avec un peu de grele
    99: {"d": 21, "n": 21},   # Orage avec beaucoup de grele
}

print(f"La table contient {len(WMO_TO_METEO)} codes WMO.")

Combien de ces codes ont vraiment une icone differente entre le jour et la nuit ?

In [ ]:
differents = [code for code, m in WMO_TO_METEO.items() if m["d"] != m["n"]]
identiques = [code for code, m in WMO_TO_METEO.items() if m["d"] == m["n"]]

print(f"Codes avec icone jour != icone nuit : {len(differents)} -> {differents}")
print(f"Codes avec icone jour == icone nuit : {len(identiques)} -> {identiques}")

On voit donc que **seuls 4 codes WMO** (0, 1, 2, 3 = ciel degage, peu nuageux, partiellement nuageux, couvert) ont vraiment deux variantes. Pour tous les autres, la meme icone est utilisee de jour comme de nuit.

## 2. La fonction `get_meteo_icon()`

C'est la traduction Python de la fonction JavaScript `getMeteoIcon()` qui se trouve dans `public/app.js` :

```javascript
function getMeteoIcon(wmoCode, daytime = true) {
  const m = WMO_TO_METEO[wmoCode];
  if (m) return daytime ? m.d : m.n;
  return daytime ? 1 : 101;
}
```

Elle fait trois choses :

1. Elle cherche le code WMO dans la table.
2. Si elle le trouve, elle renvoie le numero d'icone (jour ou nuit selon le parametre).
3. Si elle ne le trouve pas, elle renvoie l'icone du **ciel degage** par defaut (1 le jour, 101 la nuit).

In [ ]:
def get_meteo_icon(wmo_code: int, daytime: bool = True) -> int:
    """Convertit un code WMO en numero d'icone MeteoSwiss.

    Si le code est inconnu, retourne l'icone du ciel degage (1 le jour, 101 la nuit).
    """
    m = WMO_TO_METEO.get(wmo_code)
    if m is not None:
        return m["d"] if daytime else m["n"]
    return 1 if daytime else 101


## 3. Cas simple : un code connu

Open-Meteo nous renvoie le code **0** (ciel degage). On veut savoir quelle icone afficher.

In [ ]:
icone_jour = get_meteo_icon(0, daytime=True)
icone_nuit = get_meteo_icon(0, daytime=False)

print(f"Code WMO 0 (ciel degage)")
print(f"  Le jour -> icone {icone_jour}.svg (un soleil)")
print(f"  La nuit -> icone {icone_nuit}.svg (une lune)")

Le frontend va ensuite charger l'image en faisant un appel HTTP `GET /api/icon/1` ou `GET /api/icon/101` selon le cas, et c'est le backend qui sert le fichier SVG depuis `public/icons/`.

## 4. Plusieurs codes -> meme icone

Open-Meteo distingue trois intensites de bruine, mais pour l'utilisateur ce sont visuellement la meme chose. On les fait toutes pointer vers l'icone 7.

In [ ]:
for code, label in [(51, "bruine legere"), (53, "bruine moderee"), (55, "bruine dense")]:
    icone = get_meteo_icon(code, daytime=True)
    print(f"Code WMO {code} ({label}) -> icone {icone}.svg")

Les trois bruines pointent toutes vers `7.svg`. Resultat : peu importe l'intensite, l'utilisateur voit toujours le meme petit dessin de gouttes qui tombent.

Meme principe pour le brouillard (45 et 48 -> icone 25), pour la bruine verglacante (56 et 57 -> icone 26), etc.

## 5. Code inconnu -> fallback

Si Open-Meteo nous renvoie un code qui n'est pas dans la table (par exemple un nouveau code ajoute apres une mise a jour de la norme WMO), on affiche le ciel degage par defaut au lieu de planter.

In [ ]:
icone_jour = get_meteo_icon(999, daytime=True)
icone_nuit = get_meteo_icon(999, daytime=False)

print(f"Code WMO 999 (inconnu)")
print(f"  Le jour -> icone {icone_jour}.svg (fallback : soleil)")
print(f"  La nuit -> icone {icone_nuit}.svg (fallback : lune)")

## 6. La logique jour / nuit

Dans `app.js`, la fonction `isDaytime()` decide si on est de jour ou de nuit a l'endroit interroge :

```javascript
function isDaytime(data) {
  if (!data || !data.sys) return true;
  if (!data.sys.sunrise || !data.sys.sunset) return true;
  return data.dt >= data.sys.sunrise && data.dt <= data.sys.sunset;
}
```

Elle compare le timestamp actuel (`data.dt`) aux horaires de lever et coucher du soleil (`data.sys.sunrise` et `data.sys.sunset`). Ces horaires viennent en realite de l'indicateur `is_day` envoye par Open-Meteo, que le backend transforme en faux horaires de lever/coucher dans `server.py`.

Voici la traduction Python :

In [ ]:
def is_daytime(data: dict) -> bool:
    """Vrai si le timestamp data['dt'] est entre sunrise et sunset."""
    if not data or "sys" not in data:
        return True
    sys = data["sys"]
    if not sys.get("sunrise") or not sys.get("sunset"):
        return True
    return sys["sunrise"] <= data["dt"] <= sys["sunset"]


In [ ]:
# Exemple : a midi (entre lever et coucher)
data_jour = {
    "dt": 1_700_000_000,           # midi quelque part
    "sys": {
        "sunrise": 1_699_980_000,  # 5h avant
        "sunset":  1_700_020_000,  # 5h apres
    },
}

# Exemple : a minuit (en dehors)
data_nuit = {
    "dt": 1_700_050_000,           # plus tard, apres le sunset
    "sys": {
        "sunrise": 1_699_980_000,
        "sunset":  1_700_020_000,
    },
}

print(f"data_jour : is_daytime() = {is_daytime(data_jour)}")
print(f"data_nuit : is_daytime() = {is_daytime(data_nuit)}")

## 7. Exemple complet : on simule un appel a `/api/weather`

On va simuler ce qui se passe quand l'utilisateur clique sur Lausanne en plein jour, par temps de pluie moderee. Le backend renvoie une reponse JSON, et le frontend doit decider quelle icone afficher.

In [ ]:
# Reponse simulee du backend (format simplifie)
reponse_backend = {
    "coord": {"lat": 46.52, "lon": 6.63},
    "name": "Lausanne",
    "dt": 1_700_000_000,
    "sys": {
        "country": "CH",
        "sunrise": 1_699_980_000,
        "sunset":  1_700_020_000,
    },
    "main": {"temp": 8.5, "feels_like": 6.0, "humidity": 87},
    "weather": [{"id": 63, "description": "weather"}],  # 63 = pluie moderee
}

# 1) Le frontend regarde s'il fait jour
jour = is_daytime(reponse_backend)

# 2) Le frontend extrait le code WMO
code_wmo = reponse_backend["weather"][0]["id"]

# 3) Le frontend appelle get_meteo_icon()
icone = get_meteo_icon(code_wmo, daytime=jour)

print(f"Lieu      : {reponse_backend['name']}")
print(f"Code WMO  : {code_wmo} (pluie moderee)")
print(f"Il fait   : {'jour' if jour else 'nuit'}")
print(f"Icone     : {icone}.svg")
print(f"URL chargee par le navigateur : /api/icon/{icone}")

Et voila ! Le navigateur va maintenant faire une requete `GET /api/icon/8` au backend, qui lui renvoie le fichier `public/icons/8.svg` (une icone de pluie moderee).

## A retenir

- **Aucune logique cote serveur** : le backend passe le code WMO tel quel au frontend.
- **La traduction code -> icone est cote frontend** dans la table `WMO_TO_METEO`.
- **Plusieurs codes peuvent partager la meme icone** quand la difference est trop subtile pour etre visible (les trois bruines, les deux brouillards, etc.).
- **Seules 4 conditions ont une variante jour/nuit differente** (ciel degage et ses variations).
- **Un code inconnu ne plante pas** : on affiche un ciel degage par defaut.
- **Le choix jour/nuit est decide par Open-Meteo**, pas par un calcul d'heure locale cote frontend.